<h1 align="center"><font size="10">OGBN-arxiv hyperparameter optimisation</font></h1>

<h1 align="center"><font size="5">Hyperparameter Optimisation using VS Code and Colab Server</font></h1>

<div class="alert alert-info"><p>
This is the fifth of a series of notebooks for graph network EDA, training, inference, explainability and optimisation. This notebook searches the best hyperparametor based on the result from the best model obtain during training and testing. The best model to optomised is GraphSAGE+BatchNorm usin optuna algorithms on the ogbn-arxiv dataset. After the searc the best params will beused to train the network over ten seeds each, on a Colab GPU runtime driven from VS Code, or locally.
</p></div>


In [1]:
import os, sys, shutil, subprocess, threading, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
USE_DRIVE = True          # mirror the study DB and best-config files to Drive
REPO_URL = "https://github.com/Wb-az/pyg-graph-networks.git"
BRANCH = "main"

if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=True)
        DRIVE_DIR = Path("/content/drive/MyDrive/GraphNetworks")
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    repo = Path("/content/pyg-graph-networks")
    if not repo.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
    os.chdir(repo)
    print(subprocess.run(["git", "log", "-1", "--format=code at %h  %s  (%cd)"],
                         capture_output=True, text=True).stdout.strip())
else:
    USE_DRIVE = False
    here = Path.cwd()
    root = next(p for p in [here, *here.parents] if (p / "pyproject.toml").exists())
    os.chdir(root)

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("project root:", ROOT)

Mounted at /content/drive
code at 5063d2e  Change triggered by colab kernel, the content remained unchanged  (Sat Sep 12 23:30:29 2026 +0100)
project root: /content/pyg-graph-networks


In [2]:
# Colab has torch + CUDA preinstalled; add the graph stack and optuna on top.
if IN_COLAB:
    %pip install -q torch_geometric==2.8.0.post1 ogb optuna

OPTUNA_SEARCH = ROOT / "src" / "node_classification" / "optuna_search.py"
if not OPTUNA_SEARCH.exists():
    raise RuntimeError(f"{OPTUNA_SEARCH} is missing: commit, push, then rerun from the top")
_src = OPTUNA_SEARCH.read_text()
if ('suggest_int("num_layers", 2, 3)' not in _src
        or 'suggest_float("dropout", 0.1, 0.6, step=0.05)' not in _src
        or 'suggest_categorical("weight_decay",' not in _src):
    raise RuntimeError(f"{OPTUNA_SEARCH} is older than this notebook: "
                       "commit, push, then rerun from the top")
print(OPTUNA_SEARCH, "ok")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 28.9 MB/s eta 0:00:00
/content/pyg-graph-networks/src/node_classification/optuna_search.py ok


In [3]:
OPTUNA_DIR = ROOT / "outputs" / "optuna"
METRICS_DIR = ROOT / "outputs" / "metrics" / "ogbn-arxiv"
CKPT_DIR = ROOT / "outputs" / "checkpoints" / "ogbn-arxiv"
OPTUNA_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)


def sync(src: Path, dst: Path):
    """Copy a directory tree; files in src overwrite dst (dirs_exist_ok merges)."""
    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)


if USE_DRIVE:
    DRIVE_OPTUNA = DRIVE_DIR / "optuna"
    DRIVE_METRICS = DRIVE_DIR / "metrics" / "ogbn-arxiv"
    DRIVE_CKPT = DRIVE_DIR / "checkpoints" / "ogbn-arxiv"
    names = sorted(p.name for p in DRIVE_DIR.iterdir())
    dups = [n for n in names if n.startswith(("optuna (", "metrics (", "checkpoints ("))]
    if dups:
        raise RuntimeError(f"Duplicate folders in {DRIVE_DIR}: {dups}. In the Drive web UI "
                           "merge them into the folder without the suffix, delete the "
                           "copies, then rerun from the top")
    for d in (DRIVE_OPTUNA, DRIVE_METRICS, DRIVE_CKPT):
        if not d.exists():
            d.mkdir(parents=True)
    # Restore the study DB (resumable trials), earlier best-config files, and checkpoints.
    sync(DRIVE_OPTUNA, OPTUNA_DIR)
    sync(DRIVE_METRICS, METRICS_DIR)
    sync(DRIVE_CKPT, CKPT_DIR)
    print("restored from Drive:", sorted(p.name for p in OPTUNA_DIR.glob("*.db")))


def backup():
    if USE_DRIVE:
        sync(OPTUNA_DIR, DRIVE_OPTUNA)
        sync(METRICS_DIR, DRIVE_METRICS)
        sync(CKPT_DIR, DRIVE_CKPT)


def run_with_backup(fn, *args, backup_every_s=120, **kwargs):
    """Run fn(*args, **kwargs) while periodically backing up to Drive, so a
    Colab runtime cut mid-search loses at most backup_every_s of trials, not
    the whole study. The SQLite study DB persists each trial as it completes,
    so a backup mid-run always has every trial finished up to that point."""
    stop = threading.Event()

    def keep_backing_up():
        while not stop.wait(backup_every_s):
            try:
                backup()
            except OSError as err:          # Drive hiccup: try again next round
                print("backup failed:", err)

    threading.Thread(target=keep_backing_up, daemon=True).start()
    try:
        result = fn(*args, **kwargs)
    finally:
        stop.set()
        backup()
        if USE_DRIVE:
            print("backed up to", DRIVE_DIR)
    return result

restored from Drive: ['ogbn-arxiv_sagebn_cross_entropy_balanced_accuracy.db', 'ogbn-arxiv_sagebn_cross_entropy_balanced_accuracy_v3.db', 'ogbn-arxiv_sagebn_cross_entropy_balanced_accuracy_v4.db']


In [4]:
import optuna
import src.node_classification.optuna_search as optuna_search_module
from src.node_classification.optuna_search import run_study
from src.node_classification.ogb_run import run_from_flags

In [ ]:
TARGET_TRIALS = 30

STUDY_NAME = "ogbn-arxiv_sagebn_cross_entropy_balanced_accuracy"

done_so_far = 0
db_path = OPTUNA_DIR / f"{STUDY_NAME}.db"
if db_path.exists():
    existing = optuna.load_study(study_name=STUDY_NAME, storage=f"sqlite:///{db_path}")
    done_so_far = len(existing.trials)
remaining = max(0, TARGET_TRIALS - done_so_far)
print(f"{done_so_far} trial(s) already in the study; running {remaining} more "
      f"to reach {TARGET_TRIALS} total")

# One backup after the search finishes instead; a search of this size is
# short enough that losing it whole to a VM cut is a minor cost, not worth
# risking corruption of the live database to avoid.
if remaining:
    study = run_study(
        dataset="ogbn-arxiv", model="SAGEBN", loss="cross_entropy",
        metric="balanced_accuracy", n_trials=remaining, seeds=(0,),
        epochs=300, early_stop=30, study_name=STUDY_NAME,
    )
    backup()
    if USE_DRIVE:
        print("backed up to", DRIVE_DIR)
else:
    study = optuna.load_study(study_name=STUDY_NAME, storage=f"sqlite:///{db_path}")

0 trial(s) already in the study; running 30 more to reach 30 total


/usr/local/lib/python3.13/dist-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
Downloaded 0.08 GB: 100%|██████████| 81/81 [00:03<00:00, 21.89it/s]
Processing...


Extracting /content/pyg-graph-networks/data/arxiv.zip
Loading necessary files...
This might take a while.
Processing graphs...


100%|██████████| 1/1 [00:00<00:00, 11618.57it/s]


Converting graphs into PyG objects...


...

Saving...



Done!


Optuna search: SAGEBN on ogbn-arxiv  metric=val_balanced_accuracy  loss=cross_entropy  seeds=[0]  device=cuda


[I 2026-09-12 21:29:28,151] A new study created in RDB with name: ogbn-arxiv_sagebn_cross_entropy_balanced_accuracy
[I 2026-09-12 21:29:55,936] Trial 0 finished with value: 0.48474114099924226 and parameters: {'num_layers': 3, 'hidden_channels': 128, 'dropout': 0.30000000000000004, 'lr': 0.008996299860535255, 'optimizer': 'adamw', 'weight_decay': 0.0}. Best is trial 0 with value: 0.48474114099924226.
[I 2026-09-12 21:30:03,739] Trial 1 finished with value: 0.1011118214325202 and parameters: {'num_layers': 2, 'hidden_channels': 512, 'dropout': 0.5, 'lr': 0.01928028441003346, 'optimizer': 'adam', 'weight_decay': 0.01}. Best is trial 0 with value: 0.48474114099924226.
[I 2026-09-12 21:31:02,417] Trial 2 finished with value: 0.48964617527785154 and parameters: {'num_layers': 3, 'hidden_channels': 512, 'dropout': 0.35, 'lr': 0.006912670683424534, 'optimizer': 'adamw', 'weight_decay': 0.0001}. Best is trial 2 with value: 0.48964617527785154.
[I 2026-09-12 21:32:03,470] Trial 3 finished with 


Best of 30 trials (6.8 min): val_balanced_accuracy=0.5174 at epoch 171
  num_layers: 3
  hidden_channels: 512
  dropout: 0.2
  lr: 0.001550385773372729
  optimizer: adamw
  weight_decay: 0.0001

Reproduce over all seeds with:
  uv run python -m src.node_classification.ogb_run --dataset ogbn-arxiv --model SAGEBN --loss cross_entropy --num_layers 3 --hidden_channels 512 --dropout 0.20 --lr 1.55e-03 --optimizer adamw --weight_decay 0.0001 --no-dataloader
backed up to /content/drive/MyDrive/GraphNetworks


In [ ]:
print(study.best_trial.params)


{'num_layers': 3, 'hidden_channels': 512, 'dropout': 0.2, 'lr': 0.001550385773372729, 'optimizer': 'adamw', 'weight_decay': 0.0001}


In [ ]:
import pandas as pd

df = study.trials_dataframe()[["number", "value", "params_num_layers", "params_hidden_channels",
                               "params_dropout", "params_lr", "params_optimizer", "params_weight_decay"]]
df = df.rename(columns={"value": "val_balanced_accuracy"})
df


,number,val_balanced_accuracy,params_num_layers,params_hidden_channels,params_dropout,params_lr,params_optimizer,params_weight_decay
0,0,0.484741,3,128,0.30,0.008996,adamw,0.00000
1,1,0.101112,2,512,0.50,0.019280,adam,0.01000
2,2,0.489646,3,512,0.35,0.006913,adamw,0.00010
3,3,0.517436,3,512,0.20,0.001550,adamw,0.00010
4,4,0.495173,3,256,0.15,0.001456,adam,0.00100
5,5,0.073324,3,256,0.50,0.001143,adam,0.01000
6,6,0.293654,3,256,0.40,0.023588,adamw,0.00001
7,7,0.270550,3,256,0.50,0.026380,adamw,0.00050
8,8,0.289251,3,256,0.55,0.007221,adam,0.00010
9,9,0.309478,2,256,0.40,0.004299,adamw,0.00050


In [ ]:
# Confirm the search winner over all ten seeds (never report the single
# tuned-seed score). Matches the finished grid's convention: full 500 epochs,
# no early stopping, no scheduler, full batch.
best = study.best_params
flags = (
    f"--dataset ogbn-arxiv --model SAGEBN --loss cross_entropy "
    f"--num_layers {best['num_layers']} --hidden_channels {best['hidden_channels']} "
    f"--dropout {best['dropout']:.2f} --lr {best['lr']:.6f} "
    f"--optimizer {best['optimizer']} --weight_decay {best['weight_decay']:g} "
    f"--epochs 500 --early_stop 30 --no-scheduler --no-dataloader --log_steps 50"
)
print("confirming:", flags)

summary, history = run_with_backup(run_from_flags, flags)


confirming: --dataset ogbn-arxiv --model SAGEBN --loss cross_entropy --num_layers 3 --hidden_channels 512 --dropout 0.20 --lr 0.001550 --optimizer adamw --weight_decay 0.0001 --epochs 500 --early_stop 30 --no-scheduler --no-dataloader --log_steps 50
Namespace(dataset='ogbn-arxiv', model='SAGEBN', num_layers=3, hidden_channels=512, dropout=0.2, heads=8, lr=0.00155, weight_decay=0.0001, optimizer='adamw', epochs=500, early_stop=30, scheduler=False, scheduler_metric='loss', scheduler_threshold=0.005, scheduler_patience=15, loss='cross_entropy', gamma=2.0, weight_power=1.0, dataloader=False, batch_size=1024, fanout=[15, 10], balanced=False, num_workers=0, eval='full', eval_batch_size=4096, log_steps=50, resume=False, seeds=[0, 1, 2, 3, 4, 20, 42, 123, 1234, 12345])
Running SAGEBN on ogbn-arxiv dataset
Device: cuda  Features: 128  Classes: 40

Fresh run: removed 4 old sagebn_cross_entropy_h512 file(s):
  /content/pyg-graph-networks/outputs/metrics/ogbn-arxiv/sagebn_cross_entropy_h512_result

In [ ]:
print(summary)

                            mean       std    ci_low   ci_high
Test loss               0.909202  0.005177  0.905498  0.912906
Test acc                0.717470  0.002524  0.715664  0.719276
Test precision          0.585482  0.009185  0.578911  0.592052
Test recall             0.495927  0.007115  0.490838  0.501017
Test f1                 0.508954  0.005182  0.505247  0.512661
Test balanced_accuracy  0.495927  0.007115  0.490838  0.501017
Test brier              0.201442  0.001425  0.200422  0.202461
Test roc_auc            0.971947  0.000882  0.971316  0.972578
Test avg_precision      0.550513  0.004025  0.547634  0.553392
Test ece                0.036691  0.006339  0.032156  0.041225


In [5]:
import subprocess, importlib

In [6]:
# pull the fixed file onto disk (only needed if running in Colab, which cloned separately)
if IN_COLAB:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)

importlib.reload(optuna_search_module)
print("reloaded, has weight_power fix:",
      "weight_power=1.0" in Path(optuna_search_module.__file__).read_text())

reloaded, has weight_power fix: False


In [7]:
TARGET_TRIALS = 30
STUDY_NAME_WCE = "ogbn-arxiv_sagebn_weighted_cross_entropy_balanced_accuracy"

done_so_far = 0
db_path = OPTUNA_DIR / f"{STUDY_NAME_WCE}.db"
if db_path.exists():
    existing = optuna.load_study(study_name=STUDY_NAME_WCE, storage=f"sqlite:///{db_path}")
    done_so_far = len(existing.trials)
remaining = max(0, TARGET_TRIALS - done_so_far)
print(f"{done_so_far} trial(s) already in the study; running {remaining} more "
      f"to reach {TARGET_TRIALS} total")

if remaining:
    study_wce = run_study(
    dataset="ogbn-arxiv", model="SAGEBN", loss="weighted_ce",
    metric="balanced_accuracy", seeds=(0,), n_trials=remaining,
    epochs=300, early_stop=30, study_name=STUDY_NAME_WCE)
    backup()
    if USE_DRIVE:
        print("backed up to", DRIVE_DIR)
else:
    study_wce = optuna.load_study(study_name=STUDY_NAME_WCE, storage=f"sqlite:///{db_path}")


0 trial(s) already in the study; running 30 more to reach 30 total


/usr/local/lib/python3.13/dist-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


Downloaded 0.08 GB: 100%|██████████| 81/81 [00:46<00:00,  1.74it/s]
Processing...


Extracting /content/pyg-graph-networks/data/arxiv.zip
Loading necessary files...
This might take a while.
Processing graphs...


100%|██████████| 1/1 [00:00<00:00, 13400.33it/s]


...

Saving...



Done!


Optuna search: SAGEBN on ogbn-arxiv  metric=val_balanced_accuracy  loss=weighted_ce  seeds=[0]  device=cuda


[I 2026-09-12 23:07:15,490] A new study created in RDB with name: ogbn-arxiv_sagebn_weighted_cross_entropy_balanced_accuracy
[I 2026-09-12 23:07:54,217] Trial 0 finished with value: 0.5842359674585135 and parameters: {'num_layers': 3, 'hidden_channels': 128, 'dropout': 0.30000000000000004, 'lr': 0.008996299860535255, 'optimizer': 'adamw', 'weight_decay': 0.0}. Best is trial 0 with value: 0.5842359674585135.
[I 2026-09-12 23:08:12,074] Trial 1 finished with value: 0.19434127371506668 and parameters: {'num_layers': 2, 'hidden_channels': 512, 'dropout': 0.5, 'lr': 0.01928028441003346, 'optimizer': 'adam', 'weight_decay': 0.01}. Best is trial 0 with value: 0.5842359674585135.
[I 2026-09-12 23:09:57,892] Trial 2 finished with value: 0.5749697839630349 and parameters: {'num_layers': 3, 'hidden_channels': 512, 'dropout': 0.35, 'lr': 0.006912670683424534, 'optimizer': 'adamw', 'weight_decay': 0.0001}. Best is trial 0 with value: 0.5842359674585135.
[I 2026-09-12 23:11:37,622] Trial 3 finished 


Best of 30 trials (12.1 min): val_balanced_accuracy=0.5842 at epoch 114
  num_layers: 3
  hidden_channels: 128
  dropout: 0.30000000000000004
  lr: 0.008996299860535255
  optimizer: adamw
  weight_decay: 0.0

Reproduce over all seeds with:
  uv run python -m src.node_classification.ogb_run --dataset ogbn-arxiv --model SAGEBN --loss weighted_ce --num_layers 3 --hidden_channels 128 --dropout 0.30 --lr 9.00e-03 --optimizer adamw --weight_decay 0 --no-dataloader
backed up to /content/drive/MyDrive/GraphNetworks


In [8]:
print(study_wce.best_trial.params)

{'num_layers': 3, 'hidden_channels': 128, 'dropout': 0.30000000000000004, 'lr': 0.008996299860535255, 'optimizer': 'adamw', 'weight_decay': 0.0}


In [10]:
import pandas as pd

df2 = study_wce.trials_dataframe()[["number", "value", "params_num_layers", "params_hidden_channels",
                               "params_dropout", "params_lr", "params_optimizer", "params_weight_decay"]]
df2 = df2.rename(columns={"value": "val_balanced_accuracy"})
df2

,number,val_balanced_accuracy,params_num_layers,params_hidden_channels,params_dropout,params_lr,params_optimizer,params_weight_decay
0,0,0.584236,3,128,0.30,0.008996,adamw,0.00000
1,1,0.194341,2,512,0.50,0.019280,adam,0.01000
2,2,0.574970,3,512,0.35,0.006913,adamw,0.00010
3,3,0.577558,3,512,0.20,0.001550,adamw,0.00010
4,4,0.561809,3,256,0.15,0.001456,adam,0.00100
5,5,0.182681,3,256,0.50,0.001143,adam,0.01000
6,6,0.532659,3,256,0.40,0.023588,adamw,0.00001
7,7,0.491336,3,256,0.50,0.026380,adamw,0.00050
8,8,0.522796,3,256,0.55,0.007221,adam,0.00010
9,9,0.524510,2,256,0.40,0.004299,adamw,0.00050


In [12]:
# Confirm the weighted_ce search winner over all ten seeds.
best_wce = study_wce.best_params
flags_wce = (
    f"--dataset ogbn-arxiv --model SAGEBN --loss weighted_ce "
    f"--num_layers {best_wce['num_layers']} --hidden_channels {best_wce['hidden_channels']} "
    f"--dropout {best_wce['dropout']:.2f} --lr {best_wce['lr']:.6f} "
    f"--optimizer {best_wce['optimizer']} --weight_decay {best_wce['weight_decay']:g} "
    f"--epochs 500 --early_stop 30 --no-scheduler --no-dataloader --log_steps 50"
)
print("confirming:", flags_wce)

summary_wce, history_wce = run_with_backup(run_from_flags, flags_wce)

confirming: --dataset ogbn-arxiv --model SAGEBN --loss weighted_ce --num_layers 3 --hidden_channels 128 --dropout 0.30 --lr 0.008996 --optimizer adamw --weight_decay 0 --epochs 500 --early_stop 30 --no-scheduler --no-dataloader --log_steps 50
Namespace(dataset='ogbn-arxiv', model='SAGEBN', num_layers=3, hidden_channels=128, dropout=0.3, heads=8, lr=0.008996, weight_decay=0.0, optimizer='adamw', epochs=500, early_stop=30, scheduler=False, scheduler_metric='loss', scheduler_threshold=0.005, scheduler_patience=15, loss='weighted_ce', gamma=2.0, weight_power=1.0, dataloader=False, batch_size=1024, fanout=[15, 10], balanced=False, num_workers=0, eval='full', eval_batch_size=4096, log_steps=50, resume=False, seeds=[0, 1, 2, 3, 4, 20, 42, 123, 1234, 12345])
Running SAGEBN on ogbn-arxiv dataset
Device: cuda  Features: 128  Classes: 40

Fresh run: removed 4 old sagebn_weighted_ce_h128 file(s):
  /content/pyg-graph-networks/outputs/metrics/ogbn-arxiv/sagebn_weighted_ce_h128_results.csv
  /conten

In [13]:
print(summary_wce)

                            mean       std    ci_low   ci_high
Test loss               1.453618  0.013064  1.444273  1.462964
Test acc                0.609034  0.008520  0.602939  0.615129
Test precision          0.436140  0.006100  0.431777  0.440504
Test recall             0.552863  0.007793  0.547288  0.558438
Test f1                 0.453338  0.006749  0.448510  0.458166
Test balanced_accuracy  0.552863  0.007793  0.547288  0.558438
Test brier              0.268104  0.004783  0.264682  0.271525
Test roc_auc            0.961331  0.001084  0.960555  0.962106
Test avg_precision      0.488132  0.004402  0.484983  0.491281
Test ece                0.024928  0.007452  0.019597  0.030258


In [14]:
TARGET_TRIALS = 30
STUDY_NAME_WCE_P = "ogbn-arxiv_sagebn_weighted_cross_entropy_balanced_accuracy_p0.5"

done_so_far = 0
db_path = OPTUNA_DIR / f"{STUDY_NAME_WCE_P}.db"
if db_path.exists():
    existing = optuna.load_study(study_name=STUDY_NAME_WCE_P, storage=f"sqlite:///{db_path}")
    done_so_far = len(existing.trials)
remaining = max(0, TARGET_TRIALS - done_so_far)
print(f"{done_so_far} trial(s) already in the study; running {remaining} more "
      f"to reach {TARGET_TRIALS} total")

if remaining:
    study_wce_p05 = run_study(
    dataset="ogbn-arxiv", model="SAGEBN", loss="weighted_ce",
    metric="balanced_accuracy", seeds=(0,), n_trials=remaining,
    epochs=300, weight_power=0.5, early_stop=30, study_name=STUDY_NAME_WCE_P)
    backup()
    if USE_DRIVE:
        print("backed up to", DRIVE_DIR)
else:
    study_wce_p05 = optuna.load_study(study_name=STUDY_NAME_WCE_P, storage=f"sqlite:///{db_path}")


0 trial(s) already in the study; running 30 more to reach 30 total


[I 2026-09-12 23:38:44,101] A new study created in RDB with name: ogbn-arxiv_sagebn_weighted_cross_entropy_balanced_accuracy_p0.5


Optuna search: SAGEBN on ogbn-arxiv  metric=val_balanced_accuracy  loss=weighted_ce  seeds=[0]  device=cuda


[I 2026-09-12 23:39:38,083] Trial 0 finished with value: 0.5671274200113621 and parameters: {'num_layers': 3, 'hidden_channels': 128, 'dropout': 0.30000000000000004, 'lr': 0.008996299860535255, 'optimizer': 'adamw', 'weight_decay': 0.0}. Best is trial 0 with value: 0.5671274200113621.
[I 2026-09-12 23:39:55,764] Trial 1 finished with value: 0.0996605957617123 and parameters: {'num_layers': 2, 'hidden_channels': 512, 'dropout': 0.5, 'lr': 0.01928028441003346, 'optimizer': 'adam', 'weight_decay': 0.01}. Best is trial 0 with value: 0.5671274200113621.
[I 2026-09-12 23:41:43,564] Trial 2 finished with value: 0.5537634820431705 and parameters: {'num_layers': 3, 'hidden_channels': 512, 'dropout': 0.35, 'lr': 0.006912670683424534, 'optimizer': 'adamw', 'weight_decay': 0.0001}. Best is trial 0 with value: 0.5671274200113621.
[I 2026-09-12 23:43:33,924] Trial 3 finished with value: 0.5627688447802135 and parameters: {'num_layers': 3, 'hidden_channels': 512, 'dropout': 0.2, 'lr': 0.0015503857733


Best of 30 trials (13.3 min): val_balanced_accuracy=0.5671 at epoch 177
  num_layers: 3
  hidden_channels: 128
  dropout: 0.30000000000000004
  lr: 0.008996299860535255
  optimizer: adamw
  weight_decay: 0.0

Reproduce over all seeds with:
  uv run python -m src.node_classification.ogb_run --dataset ogbn-arxiv --model SAGEBN --loss weighted_ce --num_layers 3 --hidden_channels 128 --dropout 0.30 --lr 9.00e-03 --optimizer adamw --weight_decay 0 --no-dataloader --weight_power 0.5
backed up to /content/drive/MyDrive/GraphNetworks


In [15]:
print(study_wce_p05.best_trial.params)

{'num_layers': 3, 'hidden_channels': 128, 'dropout': 0.30000000000000004, 'lr': 0.008996299860535255, 'optimizer': 'adamw', 'weight_decay': 0.0}


In [16]:
import pandas as pd

df3 = study_wce_p05.trials_dataframe()[["number", "value", "params_num_layers", "params_hidden_channels",
                               "params_dropout", "params_lr", "params_optimizer", "params_weight_decay"]]
df3 = df3.rename(columns={"value": "val_balanced_accuracy"})
df3

,number,val_balanced_accuracy,params_num_layers,params_hidden_channels,params_dropout,params_lr,params_optimizer,params_weight_decay
0,0,0.567127,3,128,0.30,0.008996,adamw,0.00000
1,1,0.099661,2,512,0.50,0.019280,adam,0.01000
2,2,0.553763,3,512,0.35,0.006913,adamw,0.00010
3,3,0.562769,3,512,0.20,0.001550,adamw,0.00010
4,4,0.550734,3,256,0.15,0.001456,adam,0.00100
5,5,0.115319,3,256,0.50,0.001143,adam,0.01000
6,6,0.471304,3,256,0.40,0.023588,adamw,0.00001
7,7,0.335744,3,256,0.50,0.026380,adamw,0.00050
8,8,0.447733,3,256,0.55,0.007221,adam,0.00010
9,9,0.460967,2,256,0.40,0.004299,adamw,0.00050


In [17]:
# Confirm the weighted_ce search winner over all ten seeds.

best_wce_p05 = study_wce_p05.best_params
flags_wce_p05 = (
    f"--dataset ogbn-arxiv --model SAGEBN --loss weighted_ce --weight_power 0.5 "
    f"--num_layers {best_wce_p05['num_layers']} --hidden_channels {best_wce_p05['hidden_channels']} "
    f"--dropout {best_wce_p05['dropout']:.2f} --lr {best_wce_p05['lr']:.6f} "
    f"--optimizer {best_wce_p05['optimizer']} --weight_decay {best_wce_p05['weight_decay']:g} "
    f"--epochs 500 --early_stop 30 --no-scheduler --no-dataloader --log_steps 50"
)
print("confirming:", flags_wce_p05)

summary_wce_p05, history_wce_p05 = run_with_backup(run_from_flags, flags_wce_p05)

confirming: --dataset ogbn-arxiv --model SAGEBN --loss weighted_ce --weight_power 0.5 --num_layers 3 --hidden_channels 128 --dropout 0.30 --lr 0.008996 --optimizer adamw --weight_decay 0 --epochs 500 --early_stop 30 --no-scheduler --no-dataloader --log_steps 50
Namespace(dataset='ogbn-arxiv', model='SAGEBN', num_layers=3, hidden_channels=128, dropout=0.3, heads=8, lr=0.008996, weight_decay=0.0, optimizer='adamw', epochs=500, early_stop=30, scheduler=False, scheduler_metric='loss', scheduler_threshold=0.005, scheduler_patience=15, loss='weighted_ce', gamma=2.0, weight_power=0.5, dataloader=False, batch_size=1024, fanout=[15, 10], balanced=False, num_workers=0, eval='full', eval_batch_size=4096, log_steps=50, resume=False, seeds=[0, 1, 2, 3, 4, 20, 42, 123, 1234, 12345])
Running SAGEBN on ogbn-arxiv dataset
Device: cuda  Features: 128  Classes: 40

Seed: 0  |  epoch 050  |  train_loss=1.4027  |  train_acc=0.6526  |  val_loss=1.3086  |  val_acc=0.6870  |  lr=9.00e-03
Seed: 0  |  epoch 100

In [19]:
print(summary_wce_p05)

                            mean       std    ci_low   ci_high
Test loss               1.198832  0.018041  1.185926  1.211739
Test acc                0.683995  0.008578  0.677858  0.690131
Test precision          0.496776  0.007664  0.491293  0.502258
Test recall             0.536437  0.013443  0.526820  0.546054
Test f1                 0.501557  0.007559  0.496150  0.506965
Test balanced_accuracy  0.536437  0.013443  0.526820  0.546054
Test brier              0.219899  0.004122  0.216951  0.222848
Test roc_auc            0.968377  0.000986  0.967672  0.969083
Test avg_precision      0.524632  0.007646  0.519162  0.530101
Test ece                0.021129  0.007212  0.015970  0.026288
